AI Assistance: OpenAI ChatGPT and Anthropic's Claude were used for code debugging, code generation, code organization, and code methodological brainstorming\. All final modeling, implementation, validation, commentary, and interpretation were performed and verified by the authors\.

In [1]:
import pandas as pd
import duckdb
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path

platform = 'deepnote'

if platform == 'deepnote':
    PROCESSED_DIR = Path('data/Processed')
if platform == 'vscode':
    PROCESSED_DIR = Path('work/Processed')

# I\. Create datetime column

In [2]:
PARQUET_PATH = (PROCESSED_DIR / 'omni_minute.parquet').as_posix()

In [3]:
# Get total record count
duckdb.sql(f"""
    SELECT
        COUNT(*) AS records
    FROM '{PARQUET_PATH}'
""")

┌──────────┐
│ records  │
│  int64   │
├──────────┤
│ 13936320 │
└──────────┘

In [4]:
# Create datetime column and export
input_path = PARQUET_PATH
output_path = (PROCESSED_DIR / 'omni_with_datetime.parquet').as_posix()

Path(output_path).parent.mkdir(parents=True, exist_ok=True)

with duckdb.connect() as con:
    con.execute(
        f"""
        COPY (
            SELECT
                make_date(year, 1, 1)
                    + (day - 1) * INTERVAL '1 day'
                    + hour * INTERVAL '1 hour'
                    + minute * INTERVAL '1 minute'
                    AS datetime,
                *
            FROM read_parquet('{input_path}')
        )
        TO '{output_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        );
        """
    )

In [5]:
print('Output created:', Path(output_path).exists())
print('Size in GB:', Path(output_path).stat().st_size / 1_000_000_000)

Output created: True
Size in GB: 0.162150959


# II\. Explore data to inform train / test split

In [6]:
# Load parquet with datetime
parquet_path = (PROCESSED_DIR / 'omni_with_datetime.parquet').as_posix()

In [7]:
# Check row counts by year
con = duckdb.connect()

year_counts = con.execute(f'''
    SELECT
        year,
        COUNT(*) AS value_count
    FROM read_parquet('{parquet_path}')
    GROUP BY year
    ORDER BY year
''').fetchall()

for year, count in year_counts:
    print(year, count)

2000 527040
2001 525600
2002 525600
2003 525600
2004 527040
2005 525600
2006 525600
2007 525600
2008 527040
2009 525600
2010 525600
2011 525600
2012 527040
2013 525600
2014 525600
2015 525600
2016 527040
2017 525600
2018 525600
2019 525600
2020 527040
2021 525600
2022 525600
2023 525600
2024 527040
2025 525600
2026 260640


In [8]:
# Get breakdown of Kp levels in the most recent 3-year period
# This reduces the data to one row per 3-hour interval to avoid over-counting Kp values


with duckdb.connect() as con:
    result = con.execute(
        f"""
        WITH recent_data AS (
            SELECT
                time_bucket(INTERVAL '3 hours', datetime) AS kp_interval,
                MAX(kp_10) / 10.0 AS kp
            FROM read_parquet('{parquet_path}')
            WHERE datetime >= (
                SELECT MAX(datetime) - INTERVAL '3 years'
                FROM read_parquet('{parquet_path}')
            )
            AND kp_10 IS NOT NULL
            GROUP BY kp_interval
        )
        SELECT
            CASE
                WHEN kp < 5 THEN 'Kp < 5'
                WHEN kp >= 5 AND kp < 7 THEN '5 <= Kp < 7'
                WHEN kp >= 7 THEN 'Kp >= 7'
            END AS kp_range,
            COUNT(*) AS kp_observation_count
        FROM recent_data
        GROUP BY kp_range
        ORDER BY
            CASE kp_range
                WHEN 'Kp < 5' THEN 1
                WHEN '5 <= Kp < 7' THEN 2
                WHEN 'Kp >= 7' THEN 3
            END;
        """).fetchall()

print(result)

[('Kp < 5', 8407), ('5 <= Kp < 7', 304), ('Kp >= 7', 58)]


In [9]:
# Validate that kp_10 is constant in each 3-hour interval

with duckdb.connect() as con:
    inconsistent_intervals = con.execute(
        f"""
        WITH interval_check AS (
            SELECT
                time_bucket(INTERVAL '3 hours', datetime) AS kp_interval,
                COUNT(*) AS row_count,
                COUNT(kp_10) AS non_null_count,
                COUNT(DISTINCT kp_10) AS distinct_kp_count,
                MIN(kp_10) AS min_kp_10,
                MAX(kp_10) AS max_kp_10
            FROM read_parquet('{parquet_path}')
            GROUP BY kp_interval
        )
        SELECT *
        FROM interval_check
        WHERE distinct_kp_count > 1
        ORDER BY kp_interval;
        """
    ).fetchdf()

print(inconsistent_intervals)

Empty DataFrame
Columns: [kp_interval, row_count, non_null_count, distinct_kp_count, min_kp_10, max_kp_10]
Index: []


In [10]:
# Estimate frequency of contiguous storms, kp >= 5, in the most recent 3 years

with duckdb.connect() as con:
    storm_periods = con.execute(
        f"""
        WITH recent_data AS (
            SELECT
                time_bucket(INTERVAL '3 hours', datetime) AS kp_interval,
                MAX(kp_10) / 10.0 AS kp
            FROM read_parquet('{parquet_path}')
            WHERE datetime >= (
                SELECT MAX(datetime) - INTERVAL '3 years'
                FROM read_parquet('{parquet_path}')
            )
              AND kp_10 IS NOT NULL
            GROUP BY kp_interval
        ),

        storm_bins AS (
            SELECT
                kp_interval,
                kp,
                LAG(kp_interval) OVER (
                    ORDER BY kp_interval
                ) AS previous_storm_interval
            FROM recent_data
            WHERE kp >= 5
        ),

        storm_starts AS (
            SELECT
                kp_interval,
                kp,
                CASE
                    WHEN previous_storm_interval IS NULL
                      OR kp_interval - previous_storm_interval
                         > INTERVAL '3 hours'
                    THEN 1
                    ELSE 0
                END AS new_storm
            FROM storm_bins
        ),

        numbered_storms AS (
            SELECT
                kp_interval,
                kp,
                SUM(new_storm) OVER (
                    ORDER BY kp_interval
                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                ) AS storm_id
            FROM storm_starts
        )

        SELECT
            storm_id,
            MIN(kp_interval) AS storm_start,
            MAX(kp_interval) + INTERVAL '3 hours' AS storm_end,
            COUNT(*) AS consecutive_kp_periods,
            COUNT(*) * 3 AS storm_duration_hours,
            MAX(kp) AS peak_kp,
            CASE
                WHEN MAX(kp) >= 9 THEN 'G5'
                WHEN MAX(kp) >= 8 THEN 'G4'
                WHEN MAX(kp) >= 7 THEN 'G3'
                WHEN MAX(kp) >= 6 THEN 'G2'
                ELSE 'G1'
            END AS peak_storm_category
        FROM numbered_storms
        GROUP BY storm_id
        ORDER BY storm_start;
        """
    ).fetchall()

for storm in storm_periods:
    print(storm)

(1, datetime.datetime(2023, 8, 5, 0, 0), datetime.datetime(2023, 8, 5, 9, 0), 3, 9, 6.3, 'G2')
(2, datetime.datetime(2023, 9, 2, 12, 0), datetime.datetime(2023, 9, 2, 15, 0), 1, 3, 5.0, 'G1')
(3, datetime.datetime(2023, 9, 2, 21, 0), datetime.datetime(2023, 9, 3, 3, 0), 2, 6, 5.3, 'G1')
(4, datetime.datetime(2023, 9, 12, 12, 0), datetime.datetime(2023, 9, 12, 18, 0), 2, 6, 5.3, 'G1')
(5, datetime.datetime(2023, 9, 18, 15, 0), datetime.datetime(2023, 9, 18, 21, 0), 2, 6, 5.0, 'G1')
(6, datetime.datetime(2023, 9, 19, 0, 0), datetime.datetime(2023, 9, 19, 6, 0), 2, 6, 6.3, 'G2')
(7, datetime.datetime(2023, 9, 19, 12, 0), datetime.datetime(2023, 9, 19, 18, 0), 2, 6, 6.0, 'G2')
(8, datetime.datetime(2023, 9, 24, 18, 0), datetime.datetime(2023, 9, 25, 3, 0), 3, 9, 5.7, 'G1')
(9, datetime.datetime(2023, 9, 26, 12, 0), datetime.datetime(2023, 9, 26, 15, 0), 1, 3, 5.0, 'G1')
(10, datetime.datetime(2023, 10, 5, 3, 0), datetime.datetime(2023, 10, 5, 6, 0), 1, 3, 5.0, 'G1')
(11, datetime.datetime(

# III\. Chronologically split data into train / test

In [11]:
# Calculate exact 3-year period for test set
parquet_path = (PROCESSED_DIR / 'omni_with_datetime.parquet').as_posix()

with duckdb.connect() as con:
    result = con.execute(
        f"""
        WITH date_range AS (
            SELECT
                MAX(datetime) - INTERVAL '3 years' AS start_datetime,
                MAX(datetime) AS end_datetime
            FROM read_parquet('{parquet_path}')
        )
        SELECT
            d.start_datetime,
            d.end_datetime,
            COUNT(*) FILTER (
                WHERE o.datetime >= d.start_datetime
                  AND o.datetime <= d.end_datetime
            ) AS subset_rows,
            COUNT(*) AS total_rows,
            100.0 * COUNT(*) FILTER (
                WHERE o.datetime >= d.start_datetime
                  AND o.datetime <= d.end_datetime
            ) / COUNT(*) AS subset_percentage
        FROM read_parquet('{parquet_path}') AS o
        CROSS JOIN date_range AS d
        GROUP BY
            d.start_datetime,
            d.end_datetime;
        """
    ).fetchone()

start_datetime, end_datetime, subset_rows, total_rows, subset_percentage = result

print(f'Start date: {start_datetime}')
print(f'End date: {end_datetime}')
print(f'Rows in most recent 3 years: {subset_rows:,}')
print(f'Total rows: {total_rows:,}')
print(f'Percentage of total rows: {subset_percentage:.2f}%')

Start date: 2023-06-30 23:59:00
End date: 2026-06-30 23:59:00
Rows in most recent 3 years: 1,578,241
Total rows: 13,936,320
Percentage of total rows: 11.32%


In [12]:
# Split the data into train/test and tag each record accordingly
parquet_path = (PROCESSED_DIR / 'omni_with_datetime.parquet').as_posix()

train_path = (PROCESSED_DIR / 'omni_train_prelim.parquet').as_posix()
test_path = (PROCESSED_DIR / 'omni_test.parquet').as_posix()

with duckdb.connect() as con:
    con.execute("SET memory_limit = '4GB'")
    con.execute("SET temp_directory = '/work/duckdb_temp'")

    con.execute(
        f"""
        CREATE TEMP TABLE split_dates AS
        SELECT
            MAX(datetime) - INTERVAL '3 years' AS test_start,
            MAX(datetime) AS test_end
        FROM read_parquet('{parquet_path}');
        """
    )

    test_start, test_end = con.execute(
        """
        SELECT test_start, test_end
        FROM split_dates;
        """
    ).fetchone()

    print(f'Train set: before {test_start}')
    print(f'Test set: {test_start} through {test_end}')

    # Training data
    con.execute(
        f"""
        COPY (
            SELECT
                o.*,
                'train' AS data_split
            FROM read_parquet('{parquet_path}') AS o
            CROSS JOIN split_dates AS s
            WHERE o.datetime < s.test_start
            ORDER BY o.datetime
        )
        TO '{train_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            ROW_GROUP_SIZE 100000
        );
        """
    )

    # Test data
    con.execute(
        f"""
        COPY (
            SELECT
                o.*,
                'test' AS data_split
            FROM read_parquet('{parquet_path}') AS o
            CROSS JOIN split_dates AS s
            WHERE o.datetime >= s.test_start
              AND o.datetime <= s.test_end
            ORDER BY o.datetime
        )
        TO '{test_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            ROW_GROUP_SIZE 100000
        );
        """
    )

Train set: before 2023-06-30 23:59:00
Test set: 2023-06-30 23:59:00 through 2026-06-30 23:59:00


In [13]:
# Validate that no rows were lost
with duckdb.connect() as con:
    verification = con.execute(
        f"""
        SELECT
            (SELECT COUNT(*)
             FROM read_parquet('{parquet_path}')) AS original_rows,

            (SELECT COUNT(*)
             FROM read_parquet('{train_path}')) AS train_rows,

            (SELECT COUNT(*)
             FROM read_parquet('{test_path}')) AS test_rows,

            (SELECT COUNT(*)
             FROM read_parquet('{train_path}'))
            +
            (SELECT COUNT(*)
             FROM read_parquet('{test_path}')) AS combined_rows;
        """
    ).fetchone()

original_rows, train_rows, test_rows, combined_rows = verification

print(f'Original rows: {original_rows:,}')
print(f'Train rows: {train_rows:,}')
print(f'Test rows: {test_rows:,}')
print(f'Combined rows: {combined_rows:,}')
print(f'All rows preserved: {original_rows == combined_rows}')
print(f'Test percentage: {100 * test_rows / original_rows:.2f}%')

Original rows: 13,936,320
Train rows: 12,358,079
Test rows: 1,578,241
Combined rows: 13,936,320
All rows preserved: True
Test percentage: 11.32%


In [14]:
# Validate the date ranges after split
with duckdb.connect() as con:
    date_ranges = con.execute(
        f"""
        SELECT
            'Train' AS split,
            MIN(datetime) AS start_datetime,
            MAX(datetime) AS end_datetime,
            COUNT(*) AS row_count
        FROM read_parquet('{train_path}')

        UNION ALL

        SELECT
            'Test' AS split,
            MIN(datetime) AS start_datetime,
            MAX(datetime) AS end_datetime,
            COUNT(*) AS row_count
        FROM read_parquet('{test_path}');
        """
    ).fetchall()

for row in date_ranges:
    print(row)

('Train', datetime.datetime(2000, 1, 1, 0, 0), datetime.datetime(2023, 6, 30, 23, 58), 12358079)
('Test', datetime.datetime(2023, 6, 30, 23, 59), datetime.datetime(2026, 6, 30, 23, 59), 1578241)


In [15]:
# Verify the train/test labeling
with duckdb.connect() as con:
    checks = con.execute(
        f"""
        SELECT
            data_split,
            MIN(datetime) AS start_datetime,
            MAX(datetime) AS end_datetime,
            COUNT(*) AS row_count,
            COUNT(DISTINCT data_split) AS distinct_labels
        FROM read_parquet([
            '{train_path}',
            '{test_path}'
        ])
        GROUP BY data_split
        ORDER BY data_split;
        """
    ).fetchall()

for row in checks:
    print(row)

('test', datetime.datetime(2023, 6, 30, 23, 59), datetime.datetime(2026, 6, 30, 23, 59), 1578241, 1)
('train', datetime.datetime(2000, 1, 1, 0, 0), datetime.datetime(2023, 6, 30, 23, 58), 12358079, 1)


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a18bf7d-431c-4909-af8a-a54a62228a78' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>